In [5]:
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

In [6]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

BATCH_SIZE = 32
SEED = 42

In [7]:
dataset_path = "/workspaces/plant_disease_detection/data/raw"

In [8]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)



Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.


In [9]:
class_names = train_dataset.class_names

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)

Number of classes: 15


In [10]:
train_dataset = train_dataset.take(100)
validation_dataset = validation_dataset.take(20)

In [11]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [12]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.2),

    layers.RandomZoom(0.2),

    layers.RandomContrast(0.2)

])

In [13]:
def build_cnn_model():

    model = models.Sequential([

        layers.Input(shape=(IMG_HEIGHT,
                            IMG_WIDTH,
                            3)),

        data_augmentation,

        layers.Rescaling(1./255),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),

        layers.Dropout(0.3),

        layers.Dense(NUM_CLASSES,
                     activation='softmax')

    ])

    return model

In [14]:
model = build_cnn_model()

In [15]:
model.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

In [16]:
early_stopping = EarlyStopping(

    monitor='val_loss',

    patience=5,

    restore_best_weights=True
)

checkpoint = ModelCheckpoint(

    filepath='../artifacts/best_model.keras',

    monitor='val_accuracy',

    save_best_only=True,

    verbose=1
)

reduce_lr = ReduceLROnPlateau(

    monitor='val_loss',

    factor=0.2,

    patience=3,

    verbose=1
)

In [17]:
EPOCHS = 5

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=EPOCHS,

    callbacks=[

        early_stopping,

        checkpoint,

        reduce_lr
    ]
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1577 - loss: 2.8139
Epoch 1: val_accuracy improved from None to 0.40781, saving model to ../artifacts/best_model.keras

Epoch 1: finished saving model to ../artifacts/best_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.2128 - loss: 2.4912 - val_accuracy: 0.4078 - val_loss: 1.9796 - learning_rate: 0.0010
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3893 - loss: 1.9667
Epoch 2: val_accuracy improved from 0.40781 to 0.52500, saving model to ../artifacts/best_model.keras

Epoch 2: finished saving model to ../artifacts/best_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.4200 - loss: 1.8499 - val_accuracy: 0.5250 - val_loss: 1.4582 - learning_rate: 0.0010
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5313 - loss: 1.5171
Epoch 3: val_accuracy improved from 0.52500 to 0.58906, saving model to ../artifacts/best_model.keras

Epoch 3: finished saving model

In [18]:
import pickle

with open('../artifacts/history.pkl', 'wb') as file:
    pickle.dump(history.history, file)